## OSMNX Library

In [0]:
%pip install osmnx==1.9.4 geopandas shapely requests tenacity pycountry --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import time
import logging
import warnings
import requests
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional
 
import osmnx as ox
import geopandas as gpd
import pandas as pd
import pycountry
from shapely.geometry import LineString
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
 
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("osm_highway")

In [0]:
CFG = {
    # Date filter: only ways last-edited on or after this date
    "since_date":       "2024-01-01T00:00:00Z",
 
    # Overpass endpoint — use a mirror if the main one is overloaded
    "overpass_url":     "https://overpass-api.de/api/interpreter",
    # Fallback mirrors (used automatically on timeout):
    "overpass_mirrors": [
        "https://overpass.kumi.systems/api/interpreter",
        "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    ],
 
    # Max seconds to wait for a single country query
    "overpass_timeout": 120,
 
    # Seconds to sleep between country requests (respect rate limits)
    "sleep_between_requests": 2,
 
    # Number of parallel workers. Keep ≤ 4 to avoid Overpass bans.
    "max_workers": 3,
 
    # Output Delta table path (set to None to skip writing)
    "output_table": "default.osm_highways_2024",
 
    # Geometry CRS — 4326 for lat/lon WGS84
    "crs": "EPSG:4326",
}


In [0]:
ox.settings.log_console = False
ox.settings.use_cache   = True
ox.settings.cache_folder = "/tmp/osmnx_cache"
ox.settings.overpass_url = CFG["overpass_url"]
ox.settings.timeout      = CFG["overpass_timeout"]
 
print("✅ Configuration loaded.")
print(f"   Extracting ways edited since : {CFG['since_date']}")
print(f"   Overpass endpoint            : {CFG['overpass_url']}")

In [0]:
HIGHWAY_TAGS = [
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "residential", "living_street",
    "unclassified", "service",
    "pedestrian", "busway",
    "track", "path",
    "footway", "cycleway",
    "bridleway", "road",
]
 
# Regex-ready alternation string for Overpass QL
HIGHWAY_REGEX = "|".join(HIGHWAY_TAGS)
 

In [0]:
# Surface tag → 'paved' | 'unpaved' | 'unknown'
# ---------------------------------------------------------------------------
_PAVED_SURFACES = {
    "paved", "asphalt", "concrete", "concrete:plates", "concrete:lanes",
    "paving_stones", "sett", "unhewn_cobblestone", "cobblestone",
    "cobblestone:flattened", "metal", "wood", "tartan", "artificial_turf",
}
_UNPAVED_SURFACES = {
    "unpaved", "compacted", "fine_gravel", "gravel", "shells", "rock",
    "pebblestone", "ground", "dirt", "earth", "grass", "grass_paver",
    "mud", "sand", "woodchips", "snow", "ice", "salt", "clay",
    "caliche", "laterite",
}
_INFERRED_PAVED_HIGHWAYS = {
    "motorway", "motorway_link", "trunk", "trunk_link",
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "residential", "living_street",
    "unclassified", "service", "pedestrian", "busway",
}
_INFERRED_UNPAVED_HIGHWAYS = {"track", "path", "footway", "bridleway", "cycleway"}

In [0]:
def classify_surface(surface: Optional[str], highway: Optional[str]) -> str:
    """
    Returns 'paved', 'unpaved', or 'unknown'.
    Mirrors carto_road_surface_type() from functions.sql.
    """
    if surface:
        s = surface.lower().strip()
        if s in _PAVED_SURFACES:
            return "paved"
        if s in _UNPAVED_SURFACES:
            return "unpaved"
    # Infer from highway class when surface tag is absent or unrecognised
    if highway:
        h = highway.lower().strip()
        if h in _INFERRED_PAVED_HIGHWAYS:
            return "paved"
        if h in _INFERRED_UNPAVED_HIGHWAYS:
            return "unpaved"
    return "unknown"
 
 
# ---------------------------------------------------------------------------
# highway= tag → broad functional class
# ---------------------------------------------------------------------------
_CLASS_MAP = {
    "motorway": "motorway",    "motorway_link": "motorway",
    "trunk":    "trunk",       "trunk_link":    "trunk",
    "primary":  "primary",     "primary_link":  "primary",
    "secondary":"secondary",   "secondary_link":"secondary",
    "tertiary": "tertiary",    "tertiary_link": "tertiary",
    "residential":   "residential", "living_street": "residential",
    "unclassified":  "unclassified", "road": "unclassified",
    "service":       "service",      "busway": "service",
    "pedestrian":    "pedestrian",
    "track":         "track",
    "path":          "path",  "footway": "path",
    "cycleway":      "path",  "bridleway": "path",
}
 
def classify_highway(highway: Optional[str]) -> str:
    """Mirrors carto_highway_class() from functions.sql."""
    return _CLASS_MAP.get((highway or "").lower().strip(), "other")
 
 

In [0]:
import re as _re
 
def parse_maxspeed(value: Optional[str]) -> Optional[float]:
    """Convert a raw maxspeed= tag to km/h float, or None."""
    if not value:
        return None
    v = value.strip()
    mph_m = _re.match(r"^([\d.]+)\s*mph$", v, _re.I)
    if mph_m:
        return round(float(mph_m.group(1)) * 1.60934, 1)
    knot_m = _re.match(r"^([\d.]+)\s*knots?$", v, _re.I)
    if knot_m:
        return round(float(knot_m.group(1)) * 1.852, 1)
    num_m = _re.match(r"^([\d.]+)$", v)
    if num_m:
        return float(num_m.group(1))
    return None   # "walk", "none", "RU:urban", etc.
 
 
print("✅ Classification helpers defined.")

In [0]:
def build_country_list(iso2_filter: Optional[list] = None) -> list[dict]:
    """
    Returns list of dicts with keys: iso2, iso3, name.
    Pass iso2_filter=['DE','FR','NG'] to restrict to specific countries.
    """
    countries = []
    for c in pycountry.countries:
        if iso2_filter and c.alpha_2 not in iso2_filter:
            continue
        countries.append({
            "iso2": c.alpha_2,
            "iso3": c.alpha_3,
            "name": c.name,
        })
    return sorted(countries, key=lambda x: x["name"])
 
# ── Edit here to restrict scope during testing ──────────────────────────────
# Set to None to run all ~250 countries (takes several hours on Overpass API)
# Example: COUNTRY_FILTER = ["NG", "GH", "KE", "ZA"]
COUNTRY_FILTER = ["ZM"]
# ────────────────────────────────────────────────────────────────────────────
 
COUNTRIES = build_country_list(COUNTRY_FILTER)
print(f"✅ {len(COUNTRIES)} countries loaded.")
print("   First 5:", [c["iso2"] for c in COUNTRIES[:5]])

In [0]:

def build_overpass_query(bbox: tuple, since: str, timeout: int) -> str:
    """
    bbox  : (south, west, north, east) in WGS84 degrees
    since : ISO 8601 timestamp string, e.g. '2024-01-01T00:00:00Z'
    """
    s, w, n, e = bbox
    return f"""
[out:json][timeout:{timeout}];
(
  way["highway"~"^({HIGHWAY_REGEX})$"]
     (newer:"{since}")
     ({s},{w},{n},{e});
);
out body geom qt;
"""
 
 
def _pick_endpoint(attempt: int = 0) -> str:
    """Rotate through Overpass mirrors on repeated failures."""
    mirrors = [CFG["overpass_url"]] + CFG["overpass_mirrors"]
    return mirrors[attempt % len(mirrors)]
 
 
@retry(
    wait=wait_exponential(multiplier=2, min=4, max=60),
    stop=stop_after_attempt(4),
    retry=retry_if_exception_type((requests.exceptions.Timeout,
                                   requests.exceptions.ConnectionError)),
    reraise=True,
)
def run_overpass_query(query: str, attempt: int = 0) -> dict:
    """POST the Overpass QL query and return parsed JSON."""
    endpoint = _pick_endpoint(attempt)
    resp = requests.post(
        endpoint,
        data={"data": query},
        timeout=CFG["overpass_timeout"] + 10,
        headers={"Accept-Charset": "utf-8;q=0.7,*;q=0.7"},
    )
    resp.raise_for_status()
    return resp.json()
 
 
def overpass_elements_to_geodataframe(elements: list, country_meta: dict) -> gpd.GeoDataFrame:
    """
    Convert raw Overpass JSON elements (ways with geometry) to a GeoDataFrame.
    Each element becomes one row with all relevant OSM tags + classification columns.
    """
    rows = []
    for el in elements:
        if el.get("type") != "way":
            continue
        geom_nodes = el.get("geometry", [])
        if len(geom_nodes) < 2:
            continue
 
        try:
            coords = [(n["lon"], n["lat"]) for n in geom_nodes]
            geom = LineString(coords)
        except Exception:
            continue
 
        tags  = el.get("tags", {})
        hw    = tags.get("highway", "")
        surf  = tags.get("surface")
        ts_raw = tags.get("timestamp") or el.get("timestamp")
 
        rows.append({
            "osm_id":        el.get("id"),
            "country_iso2":  country_meta["iso2"],
            "country_iso3":  country_meta["iso3"],
            "country_name":  country_meta["name"],
            "highway":       hw,
            "highway_class": classify_highway(hw),
            "surface_tag":   surf,
            "surface_type":  classify_surface(surf, hw),
            "road_name":     tags.get("name"),
            "ref":           tags.get("ref"),
            "lanes":         tags.get("lanes"),
            "maxspeed_raw":  tags.get("maxspeed"),
            "maxspeed_kmh":  parse_maxspeed(tags.get("maxspeed")),
            "oneway":        tags.get("oneway", "no"),
            "bridge":        tags.get("bridge", "no") not in ("no", "false", "0", None),
            "tunnel":        tags.get("tunnel", "no") not in ("no", "false", "0", None),
            "access":        tags.get("access"),
            "osm_timestamp": pd.to_datetime(ts_raw, utc=True, errors="coerce"),
            "extracted_at":  pd.Timestamp.utcnow(),
            "geometry":      geom,
        })
 
    if not rows:
        return gpd.GeoDataFrame()
 
    gdf = gpd.GeoDataFrame(rows, crs=CFG["crs"])
 
    # Geodesic length in km (requires geography-aware calculation)
    gdf_proj = gdf.to_crs("EPSG:3857")
    gdf["length_m"]  = gdf_proj.geometry.length.round(2)
    gdf["length_km"] = (gdf["length_m"] / 1000).round(4)
 
    return gdf
 
 
print("✅ Overpass query builder defined.")
 

In [0]:
def get_country_bbox(country_name: str, iso2: str) -> Optional[tuple]:
    """
    Returns (south, west, north, east) bounding box for a country.
    First tries OSMnx geocoder; falls back to a hardcoded bbox dict for
    countries that Nominatim sometimes fails to resolve.
    """
    # Hardcoded fallbacks for known-problematic entries
    _FALLBACK_BBOX = {
        "AQ": (-90, -180, -60, 180),   # Antarctica
        "TW": (21.9, 119.9, 25.4, 122.1),
        "XK": (41.8, 20.0, 43.3, 21.8),  # Kosovo
        "PS": (31.2, 34.2, 32.6, 35.6),  # Palestine
    }
    if iso2 in _FALLBACK_BBOX:
        return _FALLBACK_BBOX[iso2]
 
    try:
        gdf = ox.geocoder.geocode_to_gdf(country_name)
        bounds = gdf.total_bounds   # (minx, miny, maxx, maxy)
        return (bounds[1], bounds[0], bounds[3], bounds[2])   # s,w,n,e
    except Exception as e:
        log.warning(f"  [{iso2}] Geocoding failed: {e}")
        return None
 
 
def extract_country(country: dict, since: str = CFG["since_date"]) -> gpd.GeoDataFrame:
    """
    Full extraction pipeline for one country.
    Returns a GeoDataFrame (empty if no data or on error).
    """
    iso2  = country["iso2"]
    name  = country["name"]
    log.info(f"[{iso2}] Starting: {name}")
 
    # 1. Get bounding box
    bbox = get_country_bbox(name, iso2)
    if bbox is None:
        log.warning(f"[{iso2}] Skipping — could not resolve bbox.")
        return gpd.GeoDataFrame()
 
    s, w, n, e = bbox
    log.info(f"[{iso2}] BBox: S={s:.2f} W={w:.2f} N={n:.2f} E={e:.2f}")
 
    # 2. Build & run Overpass query
    query = build_overpass_query(bbox, since, CFG["overpass_timeout"])
    try:
        data = run_overpass_query(query)
    except Exception as e:
        log.error(f"[{iso2}] Overpass query failed: {e}")
        return gpd.GeoDataFrame()
 
    elements = data.get("elements", [])
    log.info(f"[{iso2}] Overpass returned {len(elements)} elements.")
 
    if not elements:
        return gpd.GeoDataFrame()
 
    # 3. Convert to GeoDataFrame + classify
    gdf = overpass_elements_to_geodataframe(elements, country)
    log.info(f"[{iso2}] Parsed {len(gdf)} valid road segments.")
 
    # 4. Respect Overpass rate limit
    time.sleep(CFG["sleep_between_requests"])
 
    return gdf
 
 
print("✅ Per-country extraction function defined.")

In [0]:
all_gdfs: list[gpd.GeoDataFrame] = []
failed_countries: list[str] = []
skipped_countries: list[str] = []
 
print(f"Starting extraction for {len(COUNTRIES)} countries...")
print(f"Parallel workers : {CFG['max_workers']}")
print(f"Since date       : {CFG['since_date']}")
print("-" * 60)
 
start_ts = time.time()
 
with ThreadPoolExecutor(max_workers=CFG["max_workers"]) as executor:
    future_to_country = {
        executor.submit(extract_country, c): c
        for c in COUNTRIES
    }
 
    for i, future in enumerate(as_completed(future_to_country), 1):
        country = future_to_country[future]
        iso2    = country["iso2"]
        try:
            gdf = future.result()
            if gdf is not None and len(gdf) > 0:
                all_gdfs.append(gdf)
                print(f"  [{i:>3}/{len(COUNTRIES)}] ✅ {iso2:2s}  {len(gdf):>7,} segments")
            else:
                skipped_countries.append(iso2)
                print(f"  [{i:>3}/{len(COUNTRIES)}] ⚠️  {iso2:2s}  no data")
        except Exception as exc:
            failed_countries.append(iso2)
            print(f"  [{i:>3}/{len(COUNTRIES)}] ❌ {iso2:2s}  ERROR: {exc}")
 
elapsed = time.time() - start_ts
print("-" * 60)
print(f"Extraction done in {elapsed/60:.1f} min")
print(f"  Successful : {len(all_gdfs)} countries")
print(f"  Empty      : {len(skipped_countries)} countries")
print(f"  Failed     : {len(failed_countries)} countries — {failed_countries}")
 

In [0]:

if not all_gdfs:
    raise RuntimeError("No data extracted. Check Overpass connectivity and country list.")
 
# Concatenate all country GeoDataFrames
combined: gpd.GeoDataFrame = gpd.GeoDataFrame(
    pd.concat(all_gdfs, ignore_index=True),
    crs=CFG["crs"],
)
 
print(f"Total raw segments : {len(combined):,}")
 
# ---------------------------------------------------------------------------
# Deduplication: the same OSM way can appear in multiple country bboxes
# (e.g., a road crossing a border). Keep the first occurrence per osm_id.
# ---------------------------------------------------------------------------
combined = combined.drop_duplicates(subset=["osm_id"], keep="first")
print(f"After dedup        : {len(combined):,}")
 
# ---------------------------------------------------------------------------
# Type coercions
# ---------------------------------------------------------------------------
combined["lanes"]        = pd.to_numeric(combined["lanes"], errors="coerce").astype("Int16")
combined["maxspeed_kmh"] = pd.to_numeric(combined["maxspeed_kmh"], errors="coerce")
combined["length_m"]     = combined["length_m"].astype(float)
combined["length_km"]    = combined["length_km"].astype(float)
combined["bridge"]       = combined["bridge"].astype(bool)
combined["tunnel"]       = combined["tunnel"].astype(bool)
 
# ---------------------------------------------------------------------------
# Filter: keep only rows with a parseable highway tag
# ---------------------------------------------------------------------------
combined = combined[combined["highway"].notna() & (combined["highway"] != "")]
 
print(f"After type cleanup : {len(combined):,}")
print(f"\nColumn schema:")
print(combined.dtypes)
 

In [0]:
 
summary = (
    combined
    .groupby(["country_iso2", "country_name", "highway_class", "surface_type"])
    .agg(
        road_count     = ("osm_id",       "count"),
        total_km       = ("length_km",    "sum"),
        avg_km         = ("length_km",    "mean"),
        pct_named      = ("road_name",    lambda s: round(100 * s.notna().mean(), 1)),
        pct_with_speed = ("maxspeed_kmh", lambda s: round(100 * s.notna().mean(), 1)),
    )
    .reset_index()
    .sort_values(["country_iso2", "highway_class"])
)
 
summary["total_km"] = summary["total_km"].round(2)
summary["avg_km"]   = summary["avg_km"].round(4)
 
print(f"Summary rows: {len(summary):,}")
display(summary.head(50))

In [0]:
 
surface_summary = (
    combined
    .groupby(["country_iso2", "country_name", "surface_type"])
    .agg(
        segments  = ("osm_id",    "count"),
        total_km  = ("length_km", "sum"),
    )
    .reset_index()
    .pivot_table(
        index   = ["country_iso2", "country_name"],
        columns = "surface_type",
        values  = ["segments", "total_km"],
        fill_value=0,
    )
)
 
surface_summary.columns = ["_".join(c).strip() for c in surface_summary.columns]
surface_summary = surface_summary.reset_index()
 
# Paved share %
for col in ["segments_paved", "segments_unpaved", "segments_unknown"]:
    if col not in surface_summary.columns:
        surface_summary[col] = 0
 
surface_summary["total_segments"] = (
    surface_summary["segments_paved"]
    + surface_summary["segments_unpaved"]
    + surface_summary["segments_unknown"]
)
surface_summary["pct_paved"] = (
    100 * surface_summary["segments_paved"] / surface_summary["total_segments"]
).round(1)
 
display(surface_summary.sort_values("total_km_paved", ascending=False).head(30))
 

In [0]:
combined_for_spark = combined.copy()
combined_for_spark["geometry_wkt"] = combined_for_spark["geometry"].apply(
    lambda g: g.wkt if g else None
)
combined_for_spark = combined_for_spark.drop(columns=["geometry"])
 
# Convert to Spark
spark_df = spark.createDataFrame(combined_for_spark)
spark_df.createOrReplaceTempView("osm_highways_2024")
 
print(f"Spark DataFrame created: {spark_df.count():,} rows")
spark_df.printSchema()

In [0]:
if CFG["output_table"]:
    (
        spark_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("country_iso2", "highway_class")
        .saveAsTable(CFG["output_table"])
    )
    print(f"✅ Saved to Delta table: {CFG['output_table']}")
    print("   Partitioned by: country_iso2, highway_class")
else:
    print("ℹ️  output_table is None — skipping Delta write.")

In [0]:
def refresh_countries(
    iso2_list: list[str],
    since: str = CFG["since_date"],
    target_table: str = CFG["output_table"],
) -> None:
    """
    Re-extract specific countries and MERGE into the Delta table.
    Useful for nightly / weekly incremental updates.
 
    Parameters
    ----------
    iso2_list    : e.g. ['DE', 'FR', 'PL']
    since        : new lower-bound timestamp for the Overpass query
    target_table : Delta table name to merge into
    """
    country_subset = [c for c in COUNTRIES if c["iso2"] in iso2_list]
    if not country_subset:
        print("No matching countries found.")
        return
 
    print(f"Refreshing {len(country_subset)} countries: {iso2_list}")
    fresh_gdfs = [extract_country(c, since) for c in country_subset]
    fresh_gdfs = [g for g in fresh_gdfs if len(g) > 0]
 
    if not fresh_gdfs:
        print("Nothing to refresh.")
        return
 
    fresh = gpd.GeoDataFrame(pd.concat(fresh_gdfs, ignore_index=True), crs=CFG["crs"])
    fresh["geometry_wkt"] = fresh["geometry"].apply(lambda g: g.wkt if g else None)
    fresh = fresh.drop(columns=["geometry"])
 
    fresh_spark = spark.createDataFrame(fresh)
 
    # Write to a staging table then MERGE
    fresh_spark.createOrReplaceTempView("_osm_highway_refresh_staging")
    spark.sql(f"""
        MERGE INTO {target_table} AS target
        USING _osm_highway_refresh_staging AS source
        ON target.osm_id = source.osm_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"✅ Merge complete for {iso2_list}")
 
 
# Example call (commented out — run manually as needed):
# refresh_countries(["DE", "FR"], since="2025-01-01T00:00:00Z")
 
print("✅ Notebook fully loaded.")

In [0]:
display(spark_df)